HWモデル+BSモデルでの株価SDEを考える
```math
dS_t = \mu S_t dt + \sigma S_t dW^{S} \\
dB_t = r_t B_t dt \\
dr_t = \left( \theta(t) - \alpha(t) r_t \right) + \sigma_t dW^{r} \\
dW^{S} dW^{r} = \rho dt
```
$B_t$を基準材とするQリスク中立測度でのSDEを考える
```math
dS_t = r_t S_t dt + \sigma S_t dW^{S,Q} \\
dB_t = r_t B_t dt \\
dr_t = \left( \theta(t) - \alpha(t) r_t \right) + \sigma_t dW^{r,Q} \\
dW^{S} dW^{r} = \rho dt
```
それぞれ解析解を考える。
```math
DF(t,T) = \frac{B_t}{B_T} = \exp \left( - \int_{t}^{T} r_s ds \right)\\
S_T = S_t \exp \left( \int_{t}^{T} r_s ds - \frac{1}{2} \int_{t}^{T} \sigma^2 ds + \sigma \int_{t}^{T} dW^{S,Q} \right) \\
B_T = B_t \exp \left( \int_{t}^{T} r_s ds \right)
```
$T$セトル$K$執行のフォワード取引のPV
```math
Fwd(t,T) = \text{E}^{Q} \left[ DF(t,T) (S_T - K) \right]
```
$T$セトル$K$ストライクのコールオプションのPV
```math
Call(t,K,T) = \text{E}^Q \left[ DF(t,T) \text{max}(S_T - K, 0) \right]
```



OISスワップレートのセットからディスカウントカーブを取得する
```math
\text{OIS Swap Rate} : S_i \\
\text{Swap Tenor} : [0, T_i] \\
\text{Payment Frequency} : 6M = 0.5 \\
\text{Cashflow Period} : [0, 0.5],...,[T^{j-1}, T^{j}] , ... ,[T_i - 0.5 ,T_i] \\
\text{Payment Delay} : 2D = 2/365 \\
\\
\text{Floating Side of } j : \text{Notional} \left[ \prod_k (1+ON_k \delta_k) - 1 \right] = \text{Notional} \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 \right] \\
\text{Fixed Side of } j : \text{Notional} \left[ S_i \sum_k \delta_k \right]
```
パースワップレートで満たされるのは
```math
0 = \text{E}^Q \left[ \sum_j DF(T_j + 2D) \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 - S_i \sum \delta_k \right] \right]
```
フォワードレート$-\frac{d}{dT}log(DF(T))$を補間関数として$DF(T)$を計算する関数をアウトプットとする。  
補間関数のノードとして持つのは$-\frac{d}{dT}log(DF(T_j + 2D))$
```math
DF(t) = \exp\left( \int_{0}^{t} -\frac{d}{dT} log(DF(T)) dT \right)
```

OISスワップレートのセットからディスカウントカーブを取得する
```math
\text{OIS Swap Rate} : S_i \\
\text{Swap Tenor} : [0, T_i] \\
\text{Payment Frequency} : 6M = 0.5 \\
\text{Cashflow Period} : [0, 0.5],...,[T^{j-1}, T^{j}] , ... ,[T_i - 0.5 ,T_i] \\
\text{Payment Delay} : 2D = 2/365 \\
\\
\text{Floating Side of } j : \text{Notional} \left[ \prod_k (1+ON_k \delta_k) - 1 \right] = \text{Notional} \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 \right] \\
\text{Fixed Side of } j : \text{Notional} \left[ S_i \sum_k \delta_k \right]
```
パースワップレートで満たされるのは
```math
0 = \text{E}^Q \left[ \sum_j DF(T_j + 2D) \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 - S_i \sum \delta_k \right] \right]
```
$log(DF(T)$を補間関数として$DF(T)$を計算する関数をアウトプットとする。  
補間関数のノードとして持つのは$log(DF(T_j + 2D)$である。

In [ ]:
import numpy as np
from scipy.optimize import brentq
from scipy.interpolate import CubicSpline, UnivariateSpline
from scipy.integrate import cumulative_trapezoid

# -----------------------------
# Log-linear DF interpolator
# -----------------------------
class LogDFCurve:
    def __init__(self):
        self.nodes = []      # times
        self.logdfs = []     # log DF

    def add_node(self, t, logdf):
        self.nodes.append(t)
        self.logdfs.append(logdf)
    
    def pop_node(self):
        self.nodes.pop()
        self.logdfs.pop()

    def df(self, t):
        t_arr = np.asarray(t, dtype=float)
        scalar_input = (t_arr.ndim == 0)

        t_flat = t_arr.reshape(-1)
        res = np.empty_like(t_flat)

        nodes = np.asarray(self.nodes)
        logdfs = np.asarray(self.logdfs)

        # t <= first node
        mask = t_flat <= nodes[0]
        res[mask] = np.exp(logdfs[0])

        # interpolation intervals
        for i in range(len(nodes) - 1):
            t0, t1 = nodes[i], nodes[i + 1]
            mask = (t_flat > t0) & (t_flat <= t1)
            if np.any(mask):
                w = (t_flat[mask] - t0) / (t1 - t0)
                logdf = (1 - w) * logdfs[i] + w * logdfs[i + 1]
                res[mask] = np.exp(logdf)

        # flat extrapolation (t > last node)
        mask = t_flat > nodes[-1]
        res[mask] = np.exp(logdfs[-1])

        res = res.reshape(t_arr.shape)
        return res.item() if scalar_input else res
    


class LogDFCurveSpline:
    def __init__(self):
        self.nodes = []      # times
        self.logdfs = []     # log DF
        self.spline = None

    def add_node(self, t, logdf):
        self.nodes.append(t)
        self.logdfs.append(logdf)
        if len(self.nodes) >= 2:
            self.spline = CubicSpline(self.nodes, self.logdfs, bc_type='natural', extrapolate=True)
    
    def pop_node(self):
        self.nodes.pop()
        self.logdfs.pop()
        if len(self.nodes) >= 2:
            self.spline = CubicSpline(self.nodes, self.logdfs, bc_type='natural', extrapolate=True)
        else:
            self.spline = None

    def df(self, t):
        t_arr = np.asarray(t, dtype=float)
        logdf = self.spline(t_arr)
        return np.exp(logdf)
    
    def forward_rate(self, t):
        t_arr = np.asarray(t, dtype=float)
        logdf = self.spline(t_arr)
        dlogdf_dt = self.spline.derivative(1)(t_arr)
        return -dlogdf_dt
    


    

class FwdRateCurveSpline:
    def __init__(self):
        self.nodes = []      # times
        self.logdfs = []     # log DF nodes
        self.spline = None   # spline on forward rates

    # --------------------------------
    # internal rebuild
    # --------------------------------
    def _rebuild_spline(self):
        if len(self.nodes) < 2:
            self.spline = None
            return

        t = np.asarray(self.nodes, dtype=float)
        logdf = np.asarray(self.logdfs, dtype=float)

        # forward rate nodes : f = - d logDF / dt
        fwdrates = -np.gradient(logdf, t)

        # spline on forward rates
        self.spline = CubicSpline(
            t, fwdrates,
            bc_type='natural',
            extrapolate=True
        )

    # --------------------------------
    # interface
    # --------------------------------
    def add_node(self, t, logdf):
        self.nodes.append(t)
        self.logdfs.append(logdf)
        self._rebuild_spline()

    def pop_node(self):
        self.nodes.pop()
        self.logdfs.pop()
        self._rebuild_spline()

    # --------------------------------
    # DF from fwdrate spline
    # --------------------------------
    def df(self, t):
        if self.spline is None:
            raise ValueError("Not enough nodes to build spline")

        t_arr = np.asarray(t, dtype=float)
        scalar_input = (t_arr.ndim == 0)

        t_flat = t_arr.reshape(-1)

        # integration grid = nodes + query points
        grid = np.unique(np.concatenate([self.nodes, t_flat]))
        grid.sort()

        # forward rates on grid
        f = self.spline(grid)

        # integrate f to get logDF
        logdf_grid = -cumulative_trapezoid(f, grid, initial=0.0)

        # anchor at t=0 (logDF(0)=0)
        # shift so that first node matches stored logdf
        idx0 = np.where(grid == self.nodes[0])[0][0]
        shift = self.logdfs[0] - logdf_grid[idx0]
        logdf_grid += shift

        # interpolate logDF
        logdf_interp = np.interp(t_flat, grid, logdf_grid)
        df = np.exp(logdf_interp)

        if scalar_input:
            return float(df[0])
        return df.reshape(t_arr.shape)

    # --------------------------------
    # forward rate = spline value
    # --------------------------------
    def forward_rate(self, t):
        if self.spline is None:
            raise ValueError("Not enough nodes to build spline")

        t_arr = np.asarray(t, dtype=float)
        scalar_input = (t_arr.ndim == 0)
        f = self.spline(t_arr)
        
        if scalar_input:
            return float(f)
        return f
    





def make_ois_cashflows(T, pay_freq, delay, is_bullet):
    if is_bullet:
        return [{
            "t0": 0.0,
            "t1": T,
            "pay": T + delay,
            "accrual": T
        }]
    else:
        pay_times = np.arange(pay_freq, T + 1e-12, pay_freq)
        cfs = []
        for t in pay_times:
            cfs.append({
                "t0": t - pay_freq,
                "t1": t,
                "pay": t + delay,
                "accrual": pay_freq
            })
        return cfs



# -----------------------------
# OIS bootstrap
# -----------------------------
def bootstrap_ois(
    swap_maturities,
    swap_rates,
    is_bullet_flags,   # ★追加
    pay_freq=0.5,
    delay=2/365,
    interp_target ="LogDF"
):
    if interp_target == "LogDF":
        curve = LogDFCurveSpline()
    elif interp_target == "FwdRate":
        curve = FwdRateCurveSpline()
    else:
        curve = None
    curve.add_node(0.0, 0.0)



    for T, S, is_bullet in zip(swap_maturities, swap_rates, is_bullet_flags):


        cfs = make_ois_cashflows(T, pay_freq, delay, is_bullet)

        def pv_equation(logdf_T):
            curve.add_node(T + delay, logdf_T)

            pv_float = 0.0
            pv_fixed = 0.0

            for cf in cfs:
                df0 = curve.df(cf["t0"])
                df1 = curve.df(cf["t1"])
                dfp = curve.df(cf["pay"])

                pv_float += dfp * (df0 / df1 - 1.0)
                pv_fixed += dfp * cf["accrual"]

            curve.pop_node()
            return pv_float - S * pv_fixed

        logdf = brentq(
            pv_equation,
            a=-2.0 * T,
            b=0.1,
            maxiter=100
        )

        curve.add_node(T + delay, logdf)

    return curve





import numpy as np
from scipy.optimize import minimize

# -----------------------------
# Global OIS calibration
# -----------------------------
def calibrate_ois_global(
    swap_maturities,
    swap_rates,
    is_bullet_flags,
    pay_freq=0.5,
    delay=2/365,
    interp_target="LogDF"
):
    # -----------------------------
    # time grid (node times)
    # -----------------------------
    node_times = np.array(swap_maturities) + delay
    n = len(node_times)

    # -----------------------------
    # curve factory
    # -----------------------------
    if interp_target == "LogDF":
        curve = LogDFCurveSpline()
    elif interp_target == "FwdRate":
        curve = FwdRateCurveSpline()
    else:
        raise ValueError("Unknown interp_target")

    # anchor
    curve.add_node(0.0, 0.0)

    # -----------------------------
    # prebuild cashflows
    # -----------------------------
    all_cfs = []
    for T, is_bullet in zip(swap_maturities, is_bullet_flags):
        all_cfs.append(make_ois_cashflows(T, pay_freq, delay, is_bullet))

    # -----------------------------
    # objective function
    # -----------------------------
    def objective(x):
        # rebuild curve
        curve.nodes = [0.0]
        curve.logdfs = [0.0]
        curve.spline = None

        for t, logdf in zip(node_times, x):
            curve.add_node(t, logdf)

        err2 = 0.0

        for S, cfs in zip(swap_rates, all_cfs):
            pv_float = 0.0
            pv_fixed = 0.0

            for cf in cfs:
                df0 = curve.df(cf["t0"])
                df1 = curve.df(cf["t1"])
                dfp = curve.df(cf["pay"])

                pv_float += dfp * (df0 / df1 - 1.0)
                pv_fixed += dfp * cf["accrual"]

            err = pv_float - S * pv_fixed
            err2 += err * err

        return err2

    # -----------------------------
    # constraints (monotonic DF)
    # -----------------------------
    bounds = []
    for i in range(n):
        # logDF <= 0, decreasing
        upper = 0.0
        lower = -5.0 * node_times[i]
        bounds.append((lower, upper))

    # initial guess : flat curve
    init_rate = swap_rates[0]
    x0 = -init_rate * node_times

    # -----------------------------
    # optimization
    # -----------------------------
    res = minimize(
        objective,
        x0,
        method="L-BFGS-B",
        bounds=bounds,
        options={"ftol": 1e-14, "maxiter": 500}
    )

    if not res.success:
        raise RuntimeError(res.message)

    # -----------------------------
    # build final curve
    # -----------------------------
    curve.nodes = [0.0]
    curve.logdfs = [0.0]
    curve.spline = None

    for t, logdf in zip(node_times, res.x):
        curve.add_node(t, logdf)

    return curve, res





In [ ]:
swap_maturities = [
    1/365, 7/365, 14/365, 21/365,
    1/12, 2/12, 3/12, 4/12, 5/12, 6/12,
    7/12, 8/12, 9/12, 10/12, 11/12,
    1, 15/12, 18/12,
    2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 12, 15, 20, 25, 30, 35, 40
]

swap_rates = [
    0.0072700, 0.0072760, 0.0072826, 0.0072938,
    0.0072893, 0.0072939, 0.0074357, 0.0076963, 0.0079437, 0.0081963,
    0.0084333, 0.0086607, 0.0088874, 0.0091402, 0.0093676,
    0.0096020, 0.0102250, 0.0108500,
    0.0120688, 0.0138000, 0.0150812, 0.0161250, 0.0170500,
    0.0179500, 0.0187950, 0.0196250, 0.0204660,
    0.0212625, 0.0220375, 0.0241867, 0.0271140,
    0.0289113, 0.0298972, 0.0304875, 0.0309025
]

is_bullet = [
    True, True, True, True, True, True,
    True, True, True, True, True, True,
    True, True, True, True, True,
    False, True, True,
    True, True, True, True, True, True, True, True, True, 
    True, True, True, True, True, True, True, True
]

curve_LogDF = bootstrap_ois(swap_maturities, swap_rates, is_bullet, pay_freq=0.5, delay=0/365, interp_target="LogDF")

print(curve_LogDF.df(3.0))
print(curve_LogDF.df(7.0))
print(curve_LogDF.df([1,2,3,4]))

curve_FwdRate = bootstrap_ois(swap_maturities, swap_rates, is_bullet, pay_freq=0.5, delay=0/365, interp_target="FwdRate")

print(curve_FwdRate.df(3.0))
print(curve_FwdRate.df(7.0))
print(curve_FwdRate.df([1,2,3,4]))

curve_global, res = calibrate_ois_global(
    swap_maturities,
    swap_rates,
    is_bullet,
    pay_freq=0.5,
    delay=0/365,
    interp_target="LogDF"
)

print(curve_global.df(3.0))
print(curve_global.df(7.0))
print(curve_global.df([1,2,3,4]))


2026/1/19 JSCC

| Tenor | JPY OIS (%) |
| ----: | ----------: |
|    1D |     0.72700 |
|    1W |     0.72760 |
|    2W |     0.72826 |
|    3W |     0.72938 |
|    1M |     0.72893 |
|    2M |     0.72939 |
|    3M |     0.74357 |
|    4M |     0.76963 |
|    5M |     0.79437 |
|    6M |     0.81963 |
|    7M |     0.84333 |
|    8M |     0.86607 |
|    9M |     0.88874 |
|   10M |     0.91402 |
|   11M |     0.93676 |
|    1Y |     0.96020 |
|   15M |     1.02250 |
|   18M |     1.08500 |
|    2Y |     1.20688 |
|    3Y |     1.38000 |
|    4Y |     1.50812 |
|    5Y |     1.61250 |
|    6Y |     1.70500 |
|    7Y |     1.79500 |
|    8Y |     1.87950 |
|    9Y |     1.96250 |
|   10Y |     2.04660 |
|   11Y |     2.12625 |
|   12Y |     2.20375 |
|   15Y |     2.41867 |
|   20Y |     2.71140 |
|   25Y |     2.89113 |
|   30Y |     2.98972 |
|   35Y |     3.04875 |
|   40Y |     3.09025 |


In [ ]:
swap_maturities = [
    1/365, 7/365, 14/365, 21/365,
    1/12, 2/12, 3/12, 4/12, 5/12, 6/12,
    7/12, 8/12, 9/12, 10/12, 11/12,
    1, 15/12, 18/12,
    2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 12, 15, 20, 25, 30, 35, 40
]

swap_rates = [
    0.0072700, 0.0072760, 0.0072826, 0.0072938,
    0.0072893, 0.0072939, 0.0074357, 0.0076963, 0.0079437, 0.0081963,
    0.0084333, 0.0086607, 0.0088874, 0.0091402, 0.0093676,
    0.0096020, 0.0102250, 0.0108500,
    0.0120688, 0.0138000, 0.0150812, 0.0161250, 0.0170500,
    0.0179500, 0.0187950, 0.0196250, 0.0204660,
    0.0212625, 0.0220375, 0.0241867, 0.0271140,
    0.0289113, 0.0298972, 0.0304875, 0.0309025
]

is_bullet = [
    True, True, True, True, True, True,
    True, True, True, True, True, True,
    True, True, True, True, True,
    False, True, True,
    True, True, True, True, True, True, True, True, True, 
    True, True, True, True, True, True, True, True
]



curve_FwdRate = bootstrap_ois(swap_maturities, swap_rates, is_bullet, pay_freq=0.5, delay=0/365, interp_target="FwdRate")

print(curve_FwdRate.df(3.0))
print(curve_FwdRate.df(7.0))
print(curve_FwdRate.df([1,2,3,4]))




curve_FwdRate = bootstrap_ois(swap_maturities, swap_rates, is_bullet, pay_freq=0.5, delay=0/365, interp_target="FwdRate")


grid = np.linspace(0.0, max(swap_maturities), int(max(swap_maturities) * 50))


df_vals_FwdRate = [curve_FwdRate.df(t) for t in grid]
fwd_rate_vals_FwdRate = [curve_FwdRate.forward_rate(t) for t in grid]

In [ ]:
def term_to_years(term_str):
    if term_str.endswith("Y"):
        return float(term_str.replace("Y", ""))
    elif term_str.endswith("M"):
        return float(term_str.replace("M", "")) / 12.0
    elif term_str.endswith("D"):
        return float(term_str.replace("D", "")) / 365.0
    else:
        raise ValueError(f"Unknown term format: {term_str}")

import pandas as pd
import numpy as np
import plotly.graph_objects as go

# CSV読み込み
df = pd.read_csv("IRS.csv")

# 金利をdecimalへ（1.055 → 0.01055 にする場合）
df["終値"] = df["終値"] / 100.0

# termを年数へ変換
df["maturity"] = df["term"].apply(term_to_years)

# 日付をdatetimeへ
df["日付"] = pd.to_datetime(df["日付"])

fig = go.Figure()

for date, group in df.groupby("日付"):

    group = group.sort_values("maturity")
    
    swap_maturities = group["maturity"].values.tolist()
    swap_rates = group["終値"].values.tolist()
    
    # すべてbulletと仮定
    is_bullet = [False] * len(swap_maturities)
    
    # カーブ構築
    curve = bootstrap_ois(
        swap_maturities,
        swap_rates,
        is_bullet,
        pay_freq=0.5,
        delay=0/365,
        interp_target="FwdRate"
    )
    
    # 描画グリッド
    grid = np.linspace(0.0, max(swap_maturities), 200)
    
    fwd_rates = [curve.forward_rate(t) for t in grid]
    
    fig.add_trace(
        go.Scatter(
            x=grid,
            y=fwd_rates,
            mode="lines",
            name=str(date.date())
        )
    )

fig.update_layout(
    title="Forward Rate Curve by Date",
    xaxis_title="Maturity (Years)",
    yaxis_title="Forward Rate",
    template="plotly_white"
)

fig.show()




In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# CSV読み込み
df = pd.read_csv("IRS.csv")

# % → decimal
df["終値"] = df["終値"] * 100

# term → 年数
def term_to_years(term_str):
    if term_str.endswith("Y"):
        return float(term_str.replace("Y", ""))
    elif term_str.endswith("M"):
        return float(term_str.replace("M", "")) / 12.0
    elif term_str.endswith("D"):
        return float(term_str.replace("D", "")) / 365.0

df["maturity"] = df["term"].apply(term_to_years)

df["日付"] = pd.to_datetime(df["日付"])

# pivotテーブル作成
pivot = df.pivot_table(
    index="日付",
    columns="maturity",
    values="終値"
)

pivot = pivot.sort_index()

# ヒートマップ
fig = go.Figure(
    data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index,
        colorscale="RdBu_r",
        colorbar=dict(title="Swap Rate")
    )
)

fig.update_layout(
    title="Swap Curve Heatmap",
    xaxis_title="Maturity (Years)",
    yaxis_title="Date",
    template="plotly_white"
)

fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# term → years 変換
# =========================
def term_to_years(term_str):
    if term_str.endswith("Y"):
        return float(term_str.replace("Y", ""))
    elif term_str.endswith("M"):
        return float(term_str.replace("M", "")) / 12.0
    elif term_str.endswith("D"):
        return float(term_str.replace("D", "")) / 365.0
    else:
        raise ValueError(f"Unknown term format: {term_str}")

# =========================
# CSV読み込み
# =========================
df = pd.read_csv("IRS.csv")
df["終値"] = df["終値"] / 100.0
df["maturity"] = df["term"].apply(term_to_years)
df["日付"] = pd.to_datetime(df["日付"])

dates = sorted(df["日付"].unique())

# =========================
# Subplot作成
# =========================
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Forward Rate Curve", "Discount Factor Curve")
)

# =========================
# 全日付分トレース作成
# =========================
for i, date in enumerate(dates):

    group = df[df["日付"] == date].sort_values("maturity")

    swap_maturities = group["maturity"].values.tolist()
    swap_rates = group["終値"].values.tolist()

    is_bullet = [False] * len(swap_maturities)

    curve = bootstrap_ois(
        swap_maturities,
        swap_rates,
        is_bullet,
        pay_freq=0.5,
        delay=0/365,
        interp_target="FwdRate"
    )

    grid = np.linspace(0.0, max(swap_maturities), 200)

    fwd_rates = [curve.forward_rate(t) for t in grid]
    dfs = [curve.df(t) for t in grid]

    visible_flag = (i == 0)

    # Forward curve
    fig.add_trace(
        go.Scatter(
            x=grid,
            y=fwd_rates,
            mode="lines",
            name="Forward",
            visible=visible_flag
        ),
        row=1, col=1
    )

    # DF curve
    fig.add_trace(
        go.Scatter(
            x=grid,
            y=dfs,
            mode="lines",
            name="DF",
            visible=visible_flag
        ),
        row=2, col=1
    )

# =========================
# Slider設定
# =========================
steps = []

for i in range(len(dates)):

    step = dict(
        method="update",
        args=[
            {"visible": [False] * (2 * len(dates))},
            {"title": f"Date: {dates[i]}"}
        ],
    )

    step["args"][0]["visible"][2*i] = True
    step["args"][0]["visible"][2*i + 1] = True

    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Date: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    template="plotly_white",
    height=700,
    xaxis2_title="Maturity (Years)",
    yaxis1_title="Forward Rate",
    yaxis2_title="Discount Factor"
)

fig.show()


In [ ]:
import polars as pl
import numpy as np
import plotly.graph_objects as go

# -------------------------
# 1. CSV読み込み
# -------------------------
df = pl.read_csv("IRS.csv")

# termを年換算
def term_to_years(term_str):
    if term_str.endswith("Y"):
        return float(term_str.replace("Y", ""))
    elif term_str.endswith("M"):
        return float(term_str.replace("M", "")) / 12.0
    elif term_str.endswith("D"):
        return float(term_str.replace("D", "")) / 365.0
    else:
        raise ValueError(f"Unknown term format: {term_str}")

df = df.with_columns(
    pl.col("term").map_elements(term_to_years, return_dtype=pl.Float64).alias("maturity"),
    pl.col("終値").cast(float),
    pl.col("日付").str.strptime(pl.Date, "%Y-%m-%d")
)

# -------------------------
# 2. グリッド定義
# -------------------------
grid = np.array([1,2,3,4,5,7,10,15,20,25,30])

# 結果保存用
df_results = []
fwd_results = []

# -------------------------
# 3. 日付ごとにbootstrap
# -------------------------
for date in df.select("日付").unique().sort("日付")["日付"]:

    sub = df.filter(pl.col("日付") == date).sort("maturity")

    swap_maturities = sub["maturity"].to_list()
    swap_rates = (sub["終値"] / 100).to_list()  # %→decimal
    is_bullet = [False] * len(swap_maturities)  # 必要なら調整

    curve = bootstrap_ois(
        swap_maturities,
        swap_rates,
        is_bullet,
        pay_freq=0.5,
        delay=0.0,
        interp_target="FwdRate"
    )

    for T in grid:
        df_results.append({
            "date": date,
            "tenor": f'{T}Y',
            "term": T,
            "value": curve.df(T)
        })

        fwd_results.append({
            "date": date,
            "tenor": f'{T}Y',
            "term": T,
            "value": curve.forward_rate(T)
        })

# polars化
df_df = pl.DataFrame(df_results)
df_fwd = pl.DataFrame(fwd_results)
df_df.write_csv("df_results.csv")
df_fwd.write_csv("fwd_results.csv")

In [ ]:
import plotly.express as px

# Forward Rate
fig_fwd = px.line(
    df_fwd.to_pandas(),
    x="date",
    y="value",
    color="term",
    labels={
        "value": "Forward Rate",
        "date": "Date",
        "term": "Maturity (Years)"
    },
)

fig_fwd.update_layout(
    title="Forward Rate Curve Evolution",
    coloraxis_colorbar_title="Years"
)

fig_fwd.show()


In [ ]:
fig_df = px.line(
    df_df.to_pandas(),
    x="date",
    y="value",
    color="tenor",
    labels={
        "value": "Discount Factor",
        "date": "Date",
        "tenor": "Maturity (Years)"
    },
)

fig_df.update_layout(
    title="Discount Factor Evolution",
    coloraxis_colorbar_title="Years",
)

fig_df.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -------------------------
# 1. 年限ごとのカラー作成
# -------------------------


tenors = sorted(df_df["term"].unique())
n = len(tenors)

tenor_rank = {T: i for i, T in enumerate(tenors)}


orange_start = np.array([120, 50, 0])     # ダークオレンジ
orange_end   = np.array([255, 170, 0])    # 明るいオレンジ

def tenor_color(T):
    i = tenor_rank[T]
    w = i / (n - 1)
    rgb = (1 - w) * orange_start + w * orange_end
    r, g, b = rgb.astype(int)
    return f"rgb({r},{g},{b})"



# -------------------------
# 2. subplot作成
# -------------------------

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=["Forward Rate", "Discount Factor"],
    vertical_spacing=0.05   # ← デフォルトは0.1くらい。小さくする
)

# -------------------------
# 3. Forward Rate
# -------------------------
for T in tenors:

    sub = df_fwd.filter(pl.col("term") == T).sort("date")

    fig.add_trace(
        go.Scatter(
            x=sub["date"],
            y=sub["value"],
            mode="lines",
            name=f"{T}Y",
            line=dict(color=tenor_color(T), width=2),
            legendgroup=f"{T}Y"
        ),
        row=1, col=1
    )

# -------------------------
# 4. Discount Factor
# -------------------------
for T in tenors:

    sub = df_df.filter(pl.col("term") == T).sort("date")

    fig.add_trace(
        go.Scatter(
            x=sub["date"],
            y=sub["value"],
            mode="lines",
            name=f"{T}Y",
            line=dict(color=tenor_color(T), width=2.5),
            legendgroup=f"{T}Y",
            showlegend=False  # Forwardと重複するので非表示
        ),
        row=2, col=1
    )

# -------------------------
# 5. レイアウト
# -------------------------
fig.update_layout(
    height=800,
    title="JPY FwdRate & DF Curves",
    hovermode="x unified"
)

fig.update_layout(
    template="plotly_white",
    font=dict(family="Arial", size=14),
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)

fig.update_layout(
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#d0d0d0", family="Arial"),
    legend=dict(
        bgcolor="rgba(0,0,0,0)",
        font=dict(color="#d0d0d0")
    ),
    margin=dict(t=40, b=40, l=60, r=30)
)

# 軸スタイル
fig.update_xaxes(
    showgrid=True,
    gridcolor="#2a2f36",
    linecolor="#aaaaaa",
    zeroline=False
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="#2a2f36",
    linecolor="#aaaaaa",
    zeroline=False
)


# DFは0〜1に制限

fig.show()
